# PolOCRBench test A: pomiar SOTA v1

**Najpierw ustaw GPU:** `Runtime → Zmień typ środowiska → T4 GPU`, potem **Uruchom wszystko**. Niczego nie wgrywaj.

Zamrożone 36 stron testu A (IMPACT history_print), ścieżka zero-shot: qwen3vl, paddlevl. Plik promptu i szablony są niezmienne (`benchmarks/polocrbench/prompts/zero_shot_prompt_v1.md`).

To **pomiar**, nie trening: bez strojenia wag, adapterów, korekt i bez promocji jakichkolwiek artefaktów. Wynik: `polocrbench-sota-measurement-v1-evidence.zip` (predykcje, metryki, wersje, paragon wejścia) + kopia na Dysku.

## 1. Środowisko, przypięty kod i testy

In [ ]:
from pathlib import Path
import hashlib
import os
import subprocess
import sys
from datetime import datetime, timezone

CODE_REVISION = 'e3005a16a779d3dea02423def8ffd203403e8e2f'
IMPACT_REPOSITORY = 'PiotrSty/impact-print-v2'
IMPACT_REVISION = 'a2480fde6f15284701458ff370b81cce50dc5c2d'
IMPACT_SHA256 = '0a9ffa126029703726fc5883a8279ddccf15763c4fdd483ed9b8cc034bdb42f0'
IMPACT_PATH = 'impact-print-v2-test.tar.gz'
MODELS = ('qwen3vl', 'paddlevl')
repo = Path('/content/OCR_engine-sota')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/PiotrStyla/OCR_engine.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', CODE_REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == CODE_REVISION
python = sys.executable  # interpreter runtime'u: CUDA torch z Colaba musi zostac na miejscu
subprocess.run([str(python), '-m', 'pip', 'install', '--no-cache-dir',
                'transformers==4.57.6', 'accelerate==1.13.0', 'jiwer==4.0.0', 'bitsandbytes',
                'huggingface_hub==0.36.2', 'pytest==8.4.2', 'nbformat==5.10.4'], check=True)
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
subprocess.run([str(python), '-m', 'pytest', 'tests/test_sota_benchmark.py',
                '-q', '-p', 'no:cacheprovider'], cwd=repo, check=True)
torch_info = subprocess.check_output([str(python), '-c',
    'import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())'],
    text=True).strip()
print('TORCH', torch_info)
print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                             capture_output=True, text=True).stdout.strip() or 'brak')
assert torch_info.endswith('True'), ('Runtime bez GPU albo torch bez CUDA (patrz TORCH/GPU wyzej). '
    'Ustaw: Runtime > Zmien typ srodowiska > T4, uruchom ponownie od komorki 1.')
work = Path('/content/polocrbench-sota-v1-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
work.mkdir()
print('CODE_REVISION', CODE_REVISION)
print('WORKDIR', work)


## 2. Zamrożone wejście: 36 stron testu A

Archiwum `impact-print-v2-test` z przypiętą rewizją i sumą SHA-256; staging weryfikuje sumy wszystkich obrazów względem zamrożonego manifestu.

In [ ]:
from huggingface_hub import hf_hub_download
import json as _json
archive = Path(hf_hub_download(IMPACT_REPOSITORY, IMPACT_PATH, repo_type='dataset', revision=IMPACT_REVISION))
assert hashlib.sha256(archive.read_bytes()).hexdigest() == IMPACT_SHA256, 'Checksum archiwum nie zgadza sie.'
staged = work/'benchmark'
subprocess.run([str(python), '-m', 'training.stage_impact_benchmark', '--archive', str(archive),
                '--output', str(staged)], cwd=repo, check=True)
rows = [_json.loads(line) for line in (staged/'manifest.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(rows) == 36, f'Oczekiwano 36 stron testu A, jest {len(rows)}.'
print('INPUT_READY', staged, len(rows), 'stron')


## 3. Stos modelowy

Qwen3-VL idzie przez transformers z CUDA torch Colaba. Dla PaddleOCR-VL silnik GPU nie jest dostępny dla Python 3.13 na Colab, więc pomiar PaddleOCR-VL leci na silniku CPU — poprawny, lecz wolniejszy (decyzja i wersje są nagrane w komórce i w `run.json`).

In [ ]:
# PaddleOCR-VL: pipeline 1.6 wymaga paddlepaddle 3.x, a oficjalne kola GPU
# Paddle nie istnieja dla Python 3.13 (runtime Colaba). Budujemy osobny
# Python 3.12 przez uv i tam instalujemy silnik GPU (cu118, T4 jest obslugiwane).
# Gdyby kola GPU nie weszly, zostaje silnik CPU (poprawny, lecz ~28 min/strone).
subprocess.run([str(python), '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([str(python), '-m', 'uv', 'venv', '--python', '3.12', '/content/paddle312'], check=True)
paddle_python = '/content/paddle312/bin/python'
gpu = subprocess.run([str(python), '-m', 'uv', 'pip', 'install', '-q', '--python', paddle_python,
                      'paddlepaddle-gpu==3.3.1', '--extra-index-url',
                      'https://www.paddlepaddle.org.cn/packages/stable/cu118/',
                      'paddleocr[doc-parser]', 'jiwer', 'pillow'], capture_output=True, text=True)
if gpu.returncode != 0:
    print('Kola GPU Paddle nie weszly, ide na silnik CPU:')
    print(gpu.stderr[-1500:])
    subprocess.run([str(python), '-m', 'uv', 'pip', 'install', '-q', '--python', paddle_python,
                    'paddlepaddle==3.3.1', 'paddleocr[doc-parser]', 'jiwer', 'pillow'], check=True)
versions = subprocess.check_output([paddle_python, '-c',
    'import importlib.metadata as m; print({p: m.version(p) for p in ("paddleocr", "paddlepaddle", "paddlex")})'],
    text=True)
print('PADDLE_STACK', versions)
PYTHONS = {'qwen3vl': str(python), 'paddlevl': paddle_python}


## 4. Smoke test (1 strona na model)

In [ ]:
import json as _json
for model in MODELS:
    out = work/'smoke'/model
    command = [PYTHONS[model], '-m', 'training.run_sota_benchmark', '--model', model,
               '--benchmark', str(staged), '--output', str(out), '--limit', '1']
    result = subprocess.run(command, cwd=repo, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
    assert result.returncode == 0, f'Smoke test nie przeszedl dla {model}. Wklej blad; nie uruchamiaj pelnego pomiaru.'
    smoke_pred = _json.loads((out/'predictions.jsonl').read_text(encoding='utf-8').splitlines()[0])
    assert smoke_pred['status'] == 'ok' and smoke_pred['text'].strip(), f'Pusty/bledny smoke dla {model}: {smoke_pred}'
    print('SMOKE_OK', model)


## 5. Pełny pomiar

Każdy model w osobnym procesie (pamięć GPU zwalniana między przebiegami).

In [ ]:
import json as _json
summary = {}
for model in MODELS:
    out = work/'runs'/model
    command = [PYTHONS[model], '-m', 'training.run_sota_benchmark', '--model', model,
               '--benchmark', str(staged), '--output', str(out)]
    result = subprocess.run(command, cwd=repo)
    assert result.returncode == 0, f'Pomiar nie ukonczyl sie dla {model}.'
    score = _json.loads((out/'score.json').read_text(encoding='utf-8'))
    summary[model] = {'cer_micro': score['cer_micro'], 'wer_micro': score['wer_micro'],
                      'structure_similarity': score['structure_similarity'],
                      'errors_or_missing': score['errors_or_missing']}
    print('MODEL_RESULT', model, summary[model])
(work/'summary.json').write_text(_json.dumps(summary, indent=2), encoding='utf-8')


## 6. Podsumowanie i paczka dowodowa

Przekaż `polocrbench-sota-measurement-v1-evidence.zip` do audytu. Kopia na Dysku jest zapasowa; nie zamykaj sesji przed pobraniem.

In [ ]:
import json as _json
import shutil
from google.colab import files
evidence = work/'evidence'
evidence.mkdir()
for model in MODELS:
    for name in ('predictions.jsonl', 'score.json', 'run.json'):
        shutil.copyfile(work/'runs'/model/name, evidence/f'{model}-{name}')
shutil.copyfile(staged/'verification.json', evidence/'staged-verification.json')
shutil.copyfile(work/'summary.json', evidence/'summary.json')
(evidence/'receipt.json').write_text(_json.dumps({
    'protocol_version': 'polocrbench-sota-measurement-v1', 'code_revision': CODE_REVISION,
    'impact_repository': IMPACT_REPOSITORY, 'impact_revision': IMPACT_REVISION,
    'impact_sha256': IMPACT_SHA256, 'models': list(MODELS),
    'measurement_only': True, 'training_performed': False}, indent=2), encoding='utf-8')
zip_path = work/'polocrbench-sota-measurement-v1-evidence.zip'
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', evidence)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copyfile(zip_path, '/content/drive/MyDrive/'+zip_path.name)
    print('KOPIA_ZAPASOWA', '/content/drive/MyDrive/'+zip_path.name)
except Exception as error:
    print('Brak kopii na Dysku:', error)
print('Gotowy ZIP:', zip_path)
files.download(str(zip_path))
